<a href="https://colab.research.google.com/github/mahathirumalathi97-web/Aerial-Object-/blob/main/Aerial_Object_classification_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aerial Object Classification project

# Train / Val / Test Split

In [3]:
import os
import shutil
import random

# Define the main directory containing class folders (e.g., 'bird', 'drone')
main_data_dir = r"/content/drive/MyDrive/aerial/data set"
base_dir = "data_split"

# Explicitly define the classes to process.
# Assuming 'bird' and 'drone' are subdirectories within main_data_dir
classes = ["bird", "drone"]

split_ratio = (0.7, 0.15, 0.15)

for cls_name in classes:
    # Create destination directories for train, val, test splits for the current class
    os.makedirs(f"{base_dir}/train/{cls_name}", exist_ok=True)
    os.makedirs(f"{base_dir}/val/{cls_name}", exist_ok=True)
    os.makedirs(f"{base_dir}/test/{cls_name}", exist_ok=True)

    # Get the full path to the source directory for the current class
    source_class_dir = os.path.join(main_data_dir, cls_name)

    # List all image files in the current class directory
    images = os.listdir(source_class_dir)
    random.shuffle(images)

    # Calculate split images
    train_end = int(len(images) * split_ratio[0])
    val_end = int(len(images) * (split_ratio[0] + split_ratio[1]))
    # Loop through images and split
    for i, img in enumerate(images):
        src = os.path.join(source_class_dir, img) # Source path of the image

        if i < train_end:    # Decide destination
            dst = f"{base_dir}/train/{cls_name}/{img}"
        elif i < val_end:
            dst = f"{base_dir}/val/{cls_name}/{img}"
        else:
            dst = f"{base_dir}/test/{cls_name}/{img}"

        shutil.copy(src, dst)  # copy the images

print("Data split into train, validation, and test sets successfully!")

Data split into train, validation, and test sets successfully!


# Tensorflow Implementation

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Data Augumentation

In [5]:
IMG_SIZE = 224   #  Define constants
BATCH_SIZE = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(    # load Training dataset
    "data_split/train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(       # load val dataset
    "data_split/val",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)



Found 2073 files belonging to 2 classes.
Found 445 files belonging to 2 classes.


#   (A) Custom CNN (TF):

In [6]:
model = models.Sequential([
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='softmax')
    ])

In [7]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [8]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 19s 159ms/step - accuracy: 0.5740 - loss: 0.7248 - val_accuracy: 0.5910 - val_loss: 0.7034
Epoch 2/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 4s 65ms/step - accuracy: 0.6363 - loss: 0.6626 - val_accuracy: 0.7124 - val_loss: 0.6239
Epoch 3/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.7086 - loss: 0.5904 - val_accuracy: 0.7101 - val_loss: 0.6240
Epoch 4/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 6s 74ms/step - accuracy: 0.7284 - loss: 0.5706 - val_accuracy: 0.7416 - val_loss: 0.5339
Epoch 5/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - accuracy: 0.7414 - loss: 0.5435 - val_accuracy: 0.7303 - val_loss: 0.5414
Epoch 6/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 4s 57ms/step - accuracy: 0.7371 - loss: 0.5531 - val_accuracy: 0.7393 - val_loss: 0.5266
Epoch 7/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - accuracy: 0.7385 - loss: 0.5324 - val_accuracy: 0.7461 - val_loss: 0.5291
Epoch 8/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.7303 - loss: 0.5434 - val_accuracy: 0.7416 -

In [9]:
model.save(r"/content/drive/MyDrive/aerial/data set/tf_custom_model.h5")

# # PyTorch Implementation

#### **Imports**

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

**Device (GPU)**

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


**Data Loading**

In [13]:
IMG_SIZE = 224
BATCH_SIZE = 16

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder("data_split/train", transform=transform)
val_data   = datasets.ImageFolder("data_split/val", transform=transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_data.classes
print(class_names)

['bird', 'drone']


**Custom CNN Model**

In [14]:
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3), nn.ReLU(), nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*26*26,128),
            nn.ReLU(),
            nn.Linear(128,2)
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

**VGG16 Transfer Learning**

In [15]:
def get_vgg16():
    model = models.vgg16(pretrained=True)

    # Freeze feature layers
    for param in model.features.parameters():
        param.requires_grad = False

    # Replace classifier
    model.classifier[6] = nn.Linear(4096, 2)

    return model

**Training Function (GPU + Metrics + Best Model Saving)**

In [16]:
def train_model(model, train_loader, val_loader, epochs, model_name):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    best_acc = 0.0

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        train_loss = running_loss / len(train_loader)

        # 🔹 Validation
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total

        print(f"{model_name} | Epoch [{epoch+1}/{epochs}] "
              f"Loss: {train_loss:.4f} "
              f"Train Acc: {train_acc:.4f} "
              f"Val Acc: {val_acc:.4f}")

        # ✅ Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"{model_name}_best.pth")

    print(f"Best Val Accuracy for {model_name}: {best_acc:.4f}")

**Train Models**

#  Custom CNN

In [17]:
cnn_model = CustomCNN()
train_model(cnn_model, train_loader, val_loader, epochs=10, model_name="custom_cnn")

custom_cnn | Epoch [1/10] Loss: 0.6118 Train Acc: 0.6705 Val Acc: 0.7528
custom_cnn | Epoch [2/10] Loss: 0.4743 Train Acc: 0.7656 Val Acc: 0.7955
custom_cnn | Epoch [3/10] Loss: 0.4148 Train Acc: 0.8095 Val Acc: 0.8067
custom_cnn | Epoch [4/10] Loss: 0.3866 Train Acc: 0.8283 Val Acc: 0.8315
custom_cnn | Epoch [5/10] Loss: 0.3376 Train Acc: 0.8553 Val Acc: 0.8292
custom_cnn | Epoch [6/10] Loss: 0.3110 Train Acc: 0.8572 Val Acc: 0.8449
custom_cnn | Epoch [7/10] Loss: 0.2841 Train Acc: 0.8784 Val Acc: 0.8337
custom_cnn | Epoch [8/10] Loss: 0.2507 Train Acc: 0.8982 Val Acc: 0.8472
custom_cnn | Epoch [9/10] Loss: 0.2399 Train Acc: 0.9021 Val Acc: 0.8539
custom_cnn | Epoch [10/10] Loss: 0.2141 Train Acc: 0.9156 Val Acc: 0.8404
Best Val Accuracy for custom_cnn: 0.8539


🔹 **VGG16**

In [18]:
from torchvision import models
vgg_model = get_vgg16()
train_model(vgg_model, train_loader, val_loader, epochs=10, model_name="vgg16")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 81.7MB/s]


vgg16 | Epoch [1/10] Loss: 0.1660 Train Acc: 0.9310 Val Acc: 0.9775
vgg16 | Epoch [2/10] Loss: 0.0171 Train Acc: 0.9947 Val Acc: 0.9910
vgg16 | Epoch [3/10] Loss: 0.0087 Train Acc: 0.9966 Val Acc: 0.9753
vgg16 | Epoch [4/10] Loss: 0.0073 Train Acc: 0.9981 Val Acc: 0.9843
vgg16 | Epoch [5/10] Loss: 0.0522 Train Acc: 0.9865 Val Acc: 0.9573
vgg16 | Epoch [6/10] Loss: 0.0337 Train Acc: 0.9894 Val Acc: 0.9775
vgg16 | Epoch [7/10] Loss: 0.0035 Train Acc: 0.9986 Val Acc: 0.9775
vgg16 | Epoch [8/10] Loss: 0.0081 Train Acc: 0.9971 Val Acc: 0.9753
vgg16 | Epoch [9/10] Loss: 0.0170 Train Acc: 0.9961 Val Acc: 0.9730
vgg16 | Epoch [10/10] Loss: 0.0031 Train Acc: 0.9995 Val Acc: 0.9685
Best Val Accuracy for vgg16: 0.9910


**Test Accuracy (Final Evaluation)**

In [19]:
test_data = datasets.ImageFolder("data_split/test", transform=transform)
test_loader = DataLoader(test_data, batch_size=16)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return correct / total

**Load Best Model & Evaluate**

In [20]:
model = get_vgg16()
model.load_state_dict(torch.load("vgg16_best.pth"))
model = model.to(device)

test_acc = evaluate(model, test_loader)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.9797752808988764


# Streamlit


In [25]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-04-16 08:01:38--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-16 08:01:38--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-16T09%3A00%3A12Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-16T0

### Pytorch App

In [26]:
%%writefile pyt.py

import streamlit as st
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image

# -------------------------------
# CONFIG
# -------------------------------
IMG_SIZE = 224
CLASSES = ['bird', 'drone']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# LOAD MODEL
# -------------------------------
@st.cache_resource
def load_model():
    model = models.vgg16(pretrained=False)
    model.classifier[6] = nn.Linear(4096, 4)

    model.load_state_dict(torch.load("vgg16_best.pth", map_location=device))
    model = model.to(device)
    model.eval()
    return model

model = load_model()

# -------------------------------
# IMAGE TRANSFORM
# -------------------------------
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

# -------------------------------
# UI
# -------------------------------
st.set_page_config(page_title="Aerial Detection", layout="centered")

st.title("👁 Aerial Object Classification & Detection (VGG16 - PyTorch)")


uploaded_file = st.file_uploader("Upload Image", type=["jpg", "png", "jpeg"])

# -------------------------------
# PREDICTION
# -------------------------------
if uploaded_file:
    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded Image", use_column_width=True)

    img = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)
        confidence, pred = torch.max(probs, 1)

    predicted_class = CLASSES[pred.item()]
    confidence = confidence.item() * 100

    st.success(f"Prediction: {predicted_class}")
    st.info(f"Confidence: {confidence:.2f}%")

Overwriting pyt.py


In [27]:
!streamlit run /content/pyt.py &>/content/logs.txt &

In [28]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://charm-stress-remains-rather.trycloudflare.com
